In [1]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from llm2vec import LLM2Vec
import json
import torch
from transformers import AutoTokenizer, AutoModel, AutoConfig
from peft import PeftModel

# Loading base Mistral model, along with custom code that enables bidirectional connections in decoder-only LLMs. MNTP LoRA weights are merged into the base model.
tokenizer = AutoTokenizer.from_pretrained(
    "McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp"
)
config = AutoConfig.from_pretrained(
    "McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp", trust_remote_code=True
)
model = AutoModel.from_pretrained(
    "McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp",
    trust_remote_code=True,
    config=config,
    torch_dtype=torch.bfloat16,
    device_map="cuda" if torch.cuda.is_available() else "cpu",
)
model = PeftModel.from_pretrained(
    model,
    "McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp",
)
model = model.merge_and_unload()  # This can take several minutes on cpu

# Loading supervised model. This loads the trained LoRA weights on top of MNTP model. Hence the final weights are -- Base model + MNTP (LoRA) + supervised (LoRA).
model = PeftModel.from_pretrained(
    model, "McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp-supervised"
)

# Wrapper for encoding and pooling operations
l2v = LLM2Vec(model, tokenizer, pooling_mode="mean", max_length=512)

In [9]:
def embedding(text,model = l2v):
    em_text = model.encode(text)
    return torch.nn.functional.normalize(em_text, p=2, dim=1)

In [11]:
def get_similarity(model):
    similarity = []
    with open(f'ground_summary/{model}_test_similarity.json', 'r') as f:
        summary = json.load(f)
        for s in summary:
            g = []
            l = []
            g.append(s['ground_summary'])
            l.append(s['llm_summary'])
            score= torch.mm(embedding(g), embedding(l).transpose(0, 1))
            similarity.append(score[0])
    return similarity


In [ ]:
model = {'claude': 0,'gpt4o':0,'gpt5':0}
for m in list(model.keys()):
    sim = get_similarity(m)
    model[m] = sum(sim)/len(sim)


In [23]:
model

{'claude': 0.6667851400375366,
 'gpt4o': 0.6227278709411621,
 'gpt5': 0.6633148193359375}